# 13b — Regression gates and uncertainty — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/13.md) · [Course map](../PLAN.md) · [Project](../../projects/stage13/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Compare candidates on paired cases
- Protect safety and weak slices
- Record reproducible release decisions


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Lesson

When comparing systems, run both on the same cases and examine paired differences. An improvement in mean score can be driven by a small subset while another group regresses. Report sample size, category breakdown, uncertainty, latency, cost, and failure counts. Do not tune against the locked final holdout repeatedly.

A release gate should define thresholds before seeing results. Some conditions are hard gates, such as any cross-tenant disclosure. Others are tradeoffs, such as a small quality increase with much higher latency. Bootstrap paired differences to estimate uncertainty while preserving case pairing; if examples share a source, resample source groups instead.

Version prompts, models, data, chunking, embeddings, rerankers, graph code, labels, and metrics. Re-run evaluations after changing any of them. Save failures as new regression cases only in a development set, preserving a separate holdout for honest assessment.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
baseline = [1, 0, 1, 1]
candidate = [1, 1, 1, 0]
print([b-a for a,b in zip(baseline,candidate)])


## Exercise 1: Paired improvement

Return mean(candidate-baseline); require equal nonzero lengths.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def paired_delta(baseline, candidate):
    if not baseline or len(baseline) != len(candidate): raise ValueError("Unpaired cases")
    return sum(b-a for a,b in zip(baseline,candidate))/len(baseline)


In [ ]:
assert paired_delta([0, 1], [1, 1]) == .5
expect_error(ValueError, lambda: paired_delta([1], [1, 0]))
print("Exercise 1: checks passed")


**Why this works:** Pairing removes avoidable variation from differences in case difficulty.


## Exercise 2: Hard release gate

Require recall>=.8, groundedness>=.9, unauthorized=0, p95_ms<=2000. Missing metrics fail.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def passes_gate(metrics):
    required = {"recall", "groundedness", "unauthorized", "p95_ms"}
    if not required <= metrics.keys(): return False
    return metrics["recall"] >= .8 and metrics["groundedness"] >= .9 and metrics["unauthorized"] == 0 and metrics["p95_ms"] <= 2000


In [ ]:
assert passes_gate(dict(recall=.9, groundedness=.95, unauthorized=0, p95_ms=1000))
assert not passes_gate(dict(recall=1, groundedness=1, unauthorized=1, p95_ms=100))
assert not passes_gate({})
print("Exercise 2: checks passed")


**Why this works:** Example thresholds are teaching defaults; a real owner must choose targets based on risk and workload.


## Exercise 3: Slice report

For rows with category and correct boolean, return category→{n,accuracy}.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def slice_report(rows):
    groups = {}
    for row in rows: groups.setdefault(row["category"], []).append(row["correct"])
    return {key: {"n": len(values), "accuracy": sum(values)/len(values)} for key, values in groups.items()}


In [ ]:
assert slice_report([{"category": "numeric", "correct": True}, {"category": "numeric", "correct": False}]) == {"numeric": {"n": 2, "accuracy": .5}}
print("Exercise 3: checks passed")


**Why this works:** A perfect score on one case carries very different evidence from a perfect score on a thousand.


## Independent transfer

Build the evaluation CLI with machine-readable reports and a nonzero exit on regression. Compare chunk sizes and models while keeping labels fixed; explain uncertainty and any excluded cases.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Explain without code

1. Why predeclare thresholds?

   It reduces the temptation to redefine success after seeing a favored candidate’s results.

2. When is ordinary bootstrap inappropriate?

   When observations are dependent, such as many paraphrases from the same source or time-correlated requests.


## Reading and review

- [Primary reference 1](https://scikit-learn.org/stable/modules/model_evaluation.html)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
